# Gold: hospital_daily_summary

Builds `adb_caresync.gold.hospital_daily_summary` (one row per hospital per day with lab activity) from silver lab_results, hospitals and calendar, with a day-over-day LAG trend, per STTM `sttm_silver_to_gold.xlsx` v1.0. Full rebuild via CREATE OR REPLACE TABLE.

In [0]:
%sql
-- Grain: one row per hospital per day (days with lab activity). Driving table: silver.lab_results grouped by hospital_id, test_date
-- Note: silver.lab_results contains exact duplicate lab_result_id rows; dedup to one row per lab test before aggregating
-- LAG returns the hospital's previous summary row (previous day with activity), per STTM
CREATE OR REPLACE TABLE adb_caresync.gold.hospital_daily_summary
COMMENT 'Gold hospital daily summary: one row per hospital per day with lab activity metrics and day-over-day trend'
AS
WITH lab_results_dedup AS (
  SELECT *
  FROM adb_caresync.silver.lab_results
  QUALIFY row_number() OVER (PARTITION BY lab_result_id ORDER BY update_timestamp DESC, insert_timestamp DESC) = 1
),
daily AS (
  SELECT
    hospital_id,
    test_date                            AS summary_date,
    count(*)                             AS total_tests,
    count(DISTINCT patient_id)           AS unique_patients_tested,
    count(DISTINCT doctor_id)            AS active_doctor_count,
    count_if(result_status = 'Abnormal') AS abnormal_test_count,
    count_if(result_status = 'Critical') AS critical_test_count
  FROM lab_results_dedup
  GROUP BY hospital_id, test_date
),
with_lag AS (
  SELECT
    d.*,
    lag(d.total_tests) OVER (PARTITION BY d.hospital_id ORDER BY d.summary_date) AS prev_day_total_tests
  FROM daily d
)
SELECT
  w.hospital_id                                                        AS hospital_id,
  h.hospital_name                                                      AS hospital_name,
  w.summary_date                                                       AS summary_date,
  c.day_name                                                           AS day_name,
  c.is_weekend                                                         AS is_weekend,
  CAST(w.total_tests AS INT)                                           AS total_tests,
  CAST(w.unique_patients_tested AS INT)                                AS unique_patients_tested,
  CAST(w.active_doctor_count AS INT)                                   AS active_doctor_count,
  CAST(w.abnormal_test_count AS INT)                                   AS abnormal_test_count,
  CAST(w.critical_test_count AS INT)                                   AS critical_test_count,
  CAST(w.abnormal_test_count * 100.0 / w.total_tests AS DECIMAL(5,2))  AS abnormal_rate_pct,
  CAST(w.critical_test_count * 100.0 / w.total_tests AS DECIMAL(5,2))  AS critical_rate_pct,
  CAST(w.prev_day_total_tests AS INT)                                  AS prev_day_total_tests,
  CAST(
    CASE WHEN w.prev_day_total_tests IS NULL OR w.prev_day_total_tests = 0 THEN NULL
         ELSE (w.total_tests - w.prev_day_total_tests) * 100.0 / w.prev_day_total_tests
    END AS DECIMAL(6,2))                                               AS test_volume_change_pct
FROM with_lag w
LEFT JOIN adb_caresync.silver.hospitals h
  ON w.hospital_id = h.hospital_id
LEFT JOIN adb_caresync.silver.calendar c
  ON c.date_key = CAST(date_format(w.summary_date, 'yyyyMMdd') AS INT)

In [0]:
# Return the record count of the rebuilt gold table to the caller (e.g. job / dbutils.notebook.run)
updated_df = spark.table("adb_caresync.gold.hospital_daily_summary")
record_count = updated_df.count()
print(f"adb_caresync.gold.hospital_daily_summary record count: {record_count}")
dbutils.notebook.exit(str(record_count))